# D2 Mean line analysis. Two-dimensional cascade

## Introduction

In general, the flow in a turbomachine is **three-dimensional and unsteady**, and quite complex. However, it can be simplified to some extent, specially
in the initial stages of designing, with a **two-dimensional cascade** with **steady** flow.

A cascade is a two-dimensional model of the flow across the blades of the rotor in the mid-span position. It allows the simplified study of **aerodynamic forces**, **pressure 
and velocity distribution**, and **boundary layer** behavior.

## Learning Objectives

- Learn the basics of cascade geometry
- Learn the fundamentals of aerodynamics on airfoils
- Apply aerodynamics to the study of the two-dimensional cascade
- Learn the concept of Diffusion Factor and its influence on cascade performance

## Previous tasks (about 3 hours)

From Dixon's book (see notebook [D1](../1_Design/D1_fundamentals_and_dimensional_analysis.ipynb)), read the following sections:

- 3.2. The part of _Compressor Blade Profiles_ is interesting. The part of _Turbine Blade Profiles_ can be skipped.
- 3.4 and 3.5. Although it is about _Compressor Cascade_, it is in general applied to axial fans as well. The part after Example 3.1 (_Nominal Deflection_, _Fluid Deviation, ...) can be skipped for now
- 5.12-5.15

## Some simple questions

In this diagram of a two-dimensional cascade, identify the following elements:
- Inlet and outlet **flow velocities** $\mathbf{c}_1$ and $\mathbf{c}_2$
- The **chord** of the airfoil $l$
- The **stagger angle** $\xi$
- The **camber angle** $\theta$
- The **pitch space** $s$ (or $t$ as can be seen in some books)
- The inlet and outlet **velocity angles** $\alpha_1$ and $\alpha_2$ (or $\beta_1$ and $\beta_2$)
- The inlet and outlet **blade angles** $\alpha'_1$ and $\alpha'_2$ (or $\beta'_1$ and $\beta'_2$ as can be seen in some books)
- The **incidence** angle $i$
- The **flow deviation** angle $\delta$

![cascade geometry](../1_Design/images/D2_cascade_geometry.svg)

Explain briefly and with your words, using basic concepts of Fluid Mechanics, the meaning of the expression (3.18)
$$
D = s\Delta p_0 \cos \alpha_m
$$

Answer:

***

The section 3.5 is long and very technical, but what is the main outcome in relation with the maximum value of diffusion factor $DF$ and the pitch-chord ratio $\frac{s}{l}$ (Lieblein, 1959)? 

Answer:

***

## Tasks

We continue with the axial fan model introduced in [D1 notebook](./D1_fundamentals_and_dimensional_analysis.ipynb). Now we analyze the performance of the two-dimensional
cascade in the mean radius. 

**Important note:** Dixon is using character $\alpha$ for angles and $\mathbf{c}$ for velocities in the cascade as an absolute reference system (for instance, a _stator_). Nevertheless, here we are going to use the notation presented for instance in [Lewis, 1996](https://www-sciencedirect-com.recursos.biblioteca.upc.edu/book/monograph/9780340631911/turbomachinery-performance-analysis) in chapter 2, where all the angles are the relative ones for the rotor, $\beta$, and velocities are relative ones $\mathbf{w}$ as we did in the D1 notebook.

Questions:

- Consider that we aim for a **diffusion factor** $DF = 0.5$, in order to be sure that the fan will work in comfortable operation point, avoiding pressure loss and noise. Estimate the suitable value for pitch-chord ratio, $s/l$, or solidity in the USA $\sigma = l/s$. (equation 3.33 in Dixon and 2.28 in Lewis)
- Although in the picture in the D1 notebook the number of blades is 9, this particular model has **10 blades**. Compute the **minimum value of the chord** for each blade in the mean radius, in order to avoid flow separation.
- Now suppose, for a first approximation, that for our airfoil, there is **no head lose** and $C_D = 0$. Find the corresponding value of $C_{L,0}$ (equation 3.26a)
- Compute again $C_L$ considering a typical value of $C_D \approx 0.02$
- The value of $C_L$ should be ideally kept about or below 1.0 in order to avoid an excessive load on the profile. In order to get that, the value of $s/l$ has to be reduced, for instance by increasing the value of the chord. Find a suitable value of the chord, $l$, in order to get a $C_L \approx 1.0$.
- Estimate the required **camber angle** $\theta$ for the airfoil. Howell's rule (equation (2.32) in Lewis and (3.43) in Dixon) states that nominal deviation is 
    $$
    \delta^* = m \theta \left(\frac{s}{l}\right)^n
    $$
    where $n = 0.5$ for compressor/fan cascade.
    - The parameter $m$ is computed with equation (2.33) in Lewis:
        $$
        m = 0.23\left(\frac{2a}{l}\right)^2 + \frac{\beta_2^*}{500}
        $$
        where $\frac{a}{l}$ is 0.5 assuming a circular camber line. 
        
    - _Nominal_ deflection is defined as $\epsilon^* = \beta_1 - \beta_2$.
        Since $i = \beta_1 - \beta'_1$ and $\theta = \beta'_1 - \beta'_2$ we get $\epsilon^* = \theta - \delta^*$ with the assumption that $i \approx 0$, derive the expression
        $$
        \theta = \frac{\epsilon^*}{1 - m \left(\frac{s}{l}\right)^n}
        $$
- Estimate the **stagger angle**, $\xi = \frac{1}{2}\left(\beta'_1 + \beta'_2\right)$

Hint: In order to make it easier, it could be interesting to write a python code that performs the computation according to some parameters. Consider using
something like that (modify it if needed):

```python
import numpy as np

def compute_2Dcascade(beta_1, beta_2, DF_obj=0.5, CD=0.02, Z_blades=10, r_m=0.15):
    """
    Computes geometric and aerodynamics parameters for a 2D cascade using diffusion factor restriction as reported by Lewis (1996)
    """
    # Conversion of angles to radians
    beta_1 = np.radians(beta_1)
    beta_2 = np.radians(beta_2)

    # mean angle
    tan_beta_inf = 0.5 * (np.tan(beta_1) + np.tan(beta_2))
    beta_inf = np.arctan(tan_beta_inf)

    # Difusion term in DF
    diffusion_term = DF_obj - (1 - np.cos(beta_1) / np.cos(beta_2))
    # geometrid term in DF
    geom_term = 0.5 * np.cos(beta_1) * (np.tan(beta_1) - np.tan(beta_2))

    t_over_l = diffusion_term / geom_term
    sigma = 1.0 / t_over_l

    # Calculate the cascade pitch (t) using the number of blades and mean radius
    t = 2 * np.pi * r_m / Z_blades
    l = t / t_over_l

    CL_id = 2 * t_over_l * np.cos(beta_inf) * (np.tan(beta_1) - np.tan(beta_2))
    CL = CL_id - CD * np.tan(beta_inf)
    
    # Now we need to get back to degrees for Howell's analysis for camber computation

    beta_1 = np.degrees(beta_1)
    beta_2 = np.degrees(beta_2) 

    # Calculate deflection

    epsilon = beta_1 - beta_2

    # Howell's parameter m
    a_over_l = 0.5 # For a circular camber line
    m = 0.23 * (2 * a_over_l) ** 2 + beta_2 / 500.0
    
    n = 0.5 # Howell's parameter n for fan cascade (n = 1 for stator cascade)
    # Camber calculation
    theta = epsilon / (1 - m * (t_over_l) ** n)

    # Nominal deviation angle
    delta = m * theta * (t_over_l) ** n

    # Assuming that i = 0
    beta_blade_1 = beta_1
    beta_blade_2 = beta_2 + delta
    # Stagger angle
    xi = 0.5 * (beta_blade_1 + beta_blade_2)

    return {
        'beta_inf': np.degrees(beta_inf),
        't_over_l': t_over_l,
        'sigma': sigma,
        'pitch (mm)': t * 1000,
        'chord_length (mm)': l * 1000,
        'CL ideal': CL_id,
        'CL': CL,
        'beta_blade_1': beta_blade_1,
        'beta_blade_2': beta_blade_2,
        'Camber angle': theta,
        'Deviation angle': delta,
        'Stagger angle': xi
    }
```


In [ ]:
# Answer:



## Your project

Now perform a similar analysis for the axial fan of your project:

- Define a suitable value for the Diffusion Factor
- Estimate, according to that, a maximum value for pitch-chord ratio.
- Taken the number of blades from the examples, estimate a minimum value of the chord. Is it consistent with the examples considered?
- Estimate the suitable value for the $C_L$ of your profile in the mean radius, provided a typical value of $C_D \approx 0.02$. Is it too large? If so,
  some iterations in order to keep it about $C_L \approx 1$. 
   

In [ ]:
# Answer:

